In [ ]:
# ==============================================================================
# FINAL MULTI-MODEL, MULTI-DATASET RESEARCH CODE (FAIR COMPARISON)
# - Dataset 1: Local Microgrid (Renewable_energy_dataset.csv)
# - Dataset 2: OPSD Time Series (time_series_60min_singleindex.csv, e.g. Germany)
# - Models:
#       1) AttentionBiLSTM (Proposed, Bahdanau-style attention)
#       2) BiLSTM (no attention)
#       3) SimpleLSTM (baseline)
# - Validation: 5-Fold Rolling Origin
# - Metrics: RMSE, MAE, nRMSE, MAPE, sMAPE + stability (Std Dev)
# - Statistical Tests: Paired t-test (Attention vs Baselines) on RMSE per fold
# - All models use the SAME loss: hybrid_loss = MSE + 0.5 * MAE
# ==============================================================================

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Bidirectional,
    Input, Reshape, Dropout,
    AdditiveAttention, Lambda
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os, random, sys

# Optional: statistical tests
try:
    from scipy.stats import ttest_rel
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("SciPy not found. Statistical significance tests will be skipped. "
          "Install with: pip install scipy")

# ------------------- 1. Reproducibility -------------------
def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seeds()

# ------------------- 2. Global Config -------------------
N_STEPS_IN = 24           # Lookback window (24 hours)
N_STEPS_OUT = 12          # Forecast horizon (12 hours ahead)
TARGET_FEATURES = ['solar_pv_output', 'wind_power_output', 'grid_load_demand']
N_FOLDS = 5
LSTM_UNITS = 150

DATASETS = [
    {
        "name": "Local Microgrid Dataset",
        "type": "local",
        "file": "Renewable_energy_dataset.csv"  # in Colab: "/content/Renewable_energy_dataset.csv"
    },
    {
        "name": "OPSD Germany Dataset (DE)",
        "type": "opsd",
        "file": "time_series_60min_singleindex.csv",  # in Colab: "/content/time_series_60min_singleindex.csv"
        "region_prefix": "DE"
    }
]

# ------------------- 3. Utility Functions -------------------
def create_sequences(data, n_steps_in, n_steps_out, target_indices):
    X, y = [], []
    for i in range(len(data)):
        end_ix = i + n_steps_in
        out_end_ix = end_ix + n_steps_out
        if out_end_ix > len(data):
            break
        seq_x = data[i:end_ix, :]
        seq_y = data[end_ix:out_end_ix, target_indices]
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

def encode_cyclical(df, col, max_val):
    df = df.copy()
    sin_name = col + '_sin'
    cos_name = col + '_cos'
    df[sin_name] = np.sin(2 * np.pi * df[col] / max_val)
    df[cos_name] = np.cos(2 * np.pi * df[col] / max_val)
    df = df.drop(columns=[col])
    return df

# ---- unified loss for all models ----
def hybrid_loss(y_true, y_pred):
    mse = tf.keras.losses.MSE(y_true, y_pred)
    mae = tf.keras.losses.MAE(y_true, y_pred)
    return mse + 0.5 * mae

# ---- Attention BiLSTM with Bahdanau-style attention ----
def build_attention_bilstm(n_steps_in, n_features, n_steps_out, n_targets,
                           units=LSTM_UNITS, dropout_rate=0.2):
    """
    Bahdanau-style: query = last timestep, keys/values = full sequence
    """
    inp = Input(shape=(n_steps_in, n_features))

    x = Bidirectional(LSTM(units, activation='relu', return_sequences=True))(inp)
    x = Dropout(dropout_rate)(x)
    x = Bidirectional(LSTM(units, activation='relu', return_sequences=True))(x)

    # query = last time step (shape: (batch, 1, features))
    query = Lambda(lambda t: t[:, -1:, :])(x)
    # Additive attention over all time steps
    context = AdditiveAttention()([query, x])   # (batch, 1, features)
    context = Lambda(lambda t: tf.squeeze(t, axis=1))(context)  # (batch, features)

    out = Dense(n_steps_out * n_targets)(context)
    out = Reshape((n_steps_out, n_targets))(out)

    model = Model(inputs=inp, outputs=out)
    # *** FAIR COMPARISON: SAME LOSS ***
    model.compile(optimizer='adam', loss=hybrid_loss)
    return model

# ---- BiLSTM baseline (no attention, SAME loss) ----
def build_bilstm_no_attention(n_steps_in, n_features, n_steps_out, n_targets,
                              units=150):
    inp = Input(shape=(n_steps_in, n_features))
    x = Bidirectional(LSTM(units, activation='relu', return_sequences=True))(inp)
    x = Bidirectional(LSTM(units, activation='relu', return_sequences=False))(x)
    out = Dense(n_steps_out * n_targets)(x)
    out = Reshape((n_steps_out, n_targets))(out)
    model = Model(inp, out)
    # *** MODIFIED: use hybrid_loss instead of 'mse' ***
    model.compile(optimizer='adam', loss=hybrid_loss)
    return model

# ---- Simple LSTM baseline (SAME loss) ----
def build_simple_lstm(n_steps_in, n_features, n_steps_out, n_targets,
                      units=100):
    model = Sequential([
        LSTM(units, activation='relu', input_shape=(n_steps_in, n_features)),
        Dense(n_steps_out * n_targets),
        Reshape((n_steps_out, n_targets))
    ])
    # *** MODIFIED: use hybrid_loss instead of 'mse' ***
    model.compile(optimizer='adam', loss=hybrid_loss)
    return model

MODELS = [
    {"name": "AttentionBiLSTM", "builder": build_attention_bilstm},
    {"name": "BiLSTM", "builder": build_bilstm_no_attention},
    {"name": "SimpleLSTM", "builder": build_simple_lstm},
]

# ------------------- 4. Dataset Loaders -------------------
def load_local_dataset(file_name):
    if not os.path.exists(file_name):
        print(f"FATAL: Local dataset file '{file_name}' not found.")
        sys.exit(1)

    df = pd.read_csv(file_name, index_col='timestamp', parse_dates=True)

    cols_to_drop = [c for c in df.columns if 'predicted' in c or 'total_energy' in c]
    df = df.drop(columns=cols_to_drop, errors='ignore')

    df = df.ffill().dropna()
    return df

def load_opsd_dataset(file_name, region_prefix="DE"):
    if not os.path.exists(file_name):
        print(f"FATAL: OPSD dataset file '{file_name}' not found.")
        sys.exit(1)

    df = pd.read_csv(file_name, index_col='utc_timestamp', parse_dates=True)

    load_col = f"{region_prefix}_load_actual_entsoe_transparency"
    solar_col = f"{region_prefix}_solar_generation_actual"
    wind_candidates = [
        f"{region_prefix}_wind_generation_actual",
        f"{region_prefix}_wind_onshore_generation_actual"
    ]

    if load_col not in df.columns or solar_col not in df.columns:
        raise ValueError(
            f"OPSD dataset missing expected columns for region '{region_prefix}': "
            f"{load_col} or {solar_col} not found."
        )

    wind_col = None
    for wc in wind_candidates:
        if wc in df.columns:
            wind_col = wc
            break
    if wind_col is None:
        raise ValueError(
            f"OPSD dataset does not have wind generation column for region '{region_prefix}'. "
            f"Tried: {wind_candidates}"
        )

    df_std = pd.DataFrame(index=df.index)
    df_std['grid_load_demand'] = df[load_col]
    df_std['solar_pv_output'] = df[solar_col]
    df_std['wind_power_output'] = df[wind_col]

    df_std['hour_of_day'] = df_std.index.hour
    df_std['day_of_week'] = df_std.index.dayofweek

    df_std = df_std.ffill().dropna()
    return df_std

def prepare_features(df):
    df = df.copy()

    df['load_lag_1'] = df['grid_load_demand'].shift(1)
    df['load_lag_24'] = df['grid_load_demand'].shift(24)
    df['load_lag_168'] = df['grid_load_demand'].shift(168)
    df = df.dropna()

    df = encode_cyclical(df, 'hour_of_day', 24)
    df = encode_cyclical(df, 'day_of_week', 7)

    feature_names = list(df.columns)

    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(df)

    target_indices = [feature_names.index(col) for col in TARGET_FEATURES]

    return df.index, data_scaled, scaler, feature_names, target_indices

# ------------------- 5. Training Loop Over Datasets & Models -------------------
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
lr_sched = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=0)

for ds_conf in DATASETS:
    print("\n" + "=" * 80)
    print(f"DATASET: {ds_conf['name']}")
    print("=" * 80)

    if ds_conf["type"] == "local":
        df_raw = load_local_dataset(ds_conf["file"])
    elif ds_conf["type"] == "opsd":
        df_raw = load_opsd_dataset(ds_conf["file"], ds_conf.get("region_prefix", "DE"))
    else:
        raise ValueError(f"Unknown dataset type: {ds_conf['type']}")

    print(f"Original shape (after basic cleaning): {df_raw.shape}")

    timestamps, data_scaled, scaler, feature_names, target_indices = prepare_features(df_raw)
    n_features = data_scaled.shape[1]
    total_len = data_scaled.shape[0]

    print(f"Prepared data shape: {data_scaled.shape} (rows x features)")

    reserved_len = int(total_len * 0.3)
    train_len = total_len - reserved_len
    fold_size = max(1, reserved_len // N_FOLDS)
    anchor_index = train_len

    print(f"Total points: {total_len}")
    print(f"Reserved for validation: {reserved_len}")
    print(f"Initial training length: {train_len}")
    print(f"Per-fold test size (approx): {fold_size}")

    # metrics[model_name][target]['rmse'/'nrmse'/'mape'/'smape'] = list of fold values
    metrics = {
        m["name"]: {
            t: {"rmse": [], "nrmse": [], "mape": [], "smape": []}
            for t in TARGET_FEATURES
        }
        for m in MODELS
    }

    # Rolling Origin CV
    for fold in range(N_FOLDS):
        test_start = anchor_index + fold * fold_size
        test_end = test_start + fold_size
        if test_start >= total_len:
            print(f"  Fold {fold+1}: skipped (test_start beyond data).")
            continue
        if test_end > total_len:
            test_end = total_len

        fold_data = data_scaled[:test_end]
        X_all, y_all = create_sequences(fold_data, N_STEPS_IN, N_STEPS_OUT, target_indices)

        seq_split = test_start - N_STEPS_IN - N_STEPS_OUT
        if seq_split <= 0 or seq_split >= X_all.shape[0]:
            print(f"  Fold {fold+1}: skipped (not enough data for sequences).")
            continue

        X_train, X_test = X_all[:seq_split], X_all[seq_split:]
        y_train, y_test = y_all[:seq_split], y_all[seq_split:]

        if X_test.shape[0] < 1:
            print(f"  Fold {fold+1}: skipped (empty test set after split).")
            continue

        print(f"\n  --- Fold {fold+1}/{N_FOLDS} ---")
        print(f"    Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

        # Train each model on the same fold
        for model_conf in MODELS:
            model_name = model_conf["name"]
            builder = model_conf["builder"]

            print(f"    -> Training model: {model_name}")

            tf.keras.backend.clear_session()
            model = builder(N_STEPS_IN, n_features, N_STEPS_OUT, len(TARGET_FEATURES))

            model.fit(
                X_train, y_train,
                epochs=100,
                batch_size=64,
                validation_split=0.1,
                callbacks=[es, lr_sched],
                verbose=0
            )

            y_pred_scaled = model.predict(X_test, verbose=0)

            y_test_flat = y_test.reshape(-1, len(TARGET_FEATURES))
            y_pred_flat = y_pred_scaled.reshape(-1, len(TARGET_FEATURES))

            dummy_true = np.zeros((y_test_flat.shape[0], n_features))
            dummy_pred = np.zeros_like(dummy_true)
            dummy_true[:, target_indices] = y_test_flat
            dummy_pred[:, target_indices] = y_pred_flat

            inv_true = scaler.inverse_transform(dummy_true)[:, target_indices]
            inv_pred = scaler.inverse_transform(dummy_pred)[:, target_indices]

            print(f"      {model_name} Fold {fold+1} Results:")
            for i, t_name in enumerate(TARGET_FEATURES):
                actual = inv_true[:, i]
                pred = inv_pred[:, i]

                rmse = np.sqrt(mean_squared_error(actual, pred))
                mae = mean_absolute_error(actual, pred)

                mean_actual = np.mean(np.abs(actual)) + 1e-8
                nrmse = rmse / mean_actual

                # MAPE: ignore very small actuals to avoid explosion (esp. solar at night)
                eps = 1e-3
                mask = np.abs(actual) > eps
                if np.any(mask):
                    mape = np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100.0
                else:
                    mape = np.nan

                # sMAPE: safer for near-zero values
                smape = np.mean(
                    2.0 * np.abs(pred - actual) /
                    (np.abs(pred) + np.abs(actual) + eps)
                ) * 100.0

                metrics[model_name][t_name]["rmse"].append(rmse)
                metrics[model_name][t_name]["nrmse"].append(nrmse)
                metrics[model_name][t_name]["mape"].append(mape)
                metrics[model_name][t_name]["smape"].append(smape)

                print(f"        {t_name}: "
                      f"RMSE = {rmse:.4f}, MAE = {mae:.4f}, "
                      f"nRMSE = {nrmse:.4f}, MAPE = {mape:.2f}%, sMAPE = {smape:.2f}%")

    # Final aggregated metrics per model
    print("\n  >>> FINAL METRICS FOR DATASET:", ds_conf["name"])
    for model_conf in MODELS:
        model_name = model_conf["name"]
        print(f"\n    MODEL: {model_name}")
        for t_name in TARGET_FEATURES:
            rmse_list = metrics[model_name][t_name]["rmse"]
            nrmse_list = metrics[model_name][t_name]["nrmse"]
            mape_list = metrics[model_name][t_name]["mape"]
            smape_list = metrics[model_name][t_name]["smape"]

            if len(rmse_list) == 0:
                print(f"      {t_name}: No valid folds.")
                continue

            avg_rmse = np.mean(rmse_list)
            std_rmse = np.std(rmse_list)
            avg_nrmse = np.mean(nrmse_list)
            avg_mape = np.nanmean(mape_list)
            avg_smape = np.nanmean(smape_list)

            print(f"      {t_name.upper()}")
            print(f"        Average RMSE       : {avg_rmse:.4f}")
            print(f"        Stability (Std Dev): {std_rmse:.4f}")
            print(f"        Average nRMSE      : {avg_nrmse:.4f}")
            print(f"        Average MAPE       : {avg_mape:.2f}%")
            print(f"        Average sMAPE      : {avg_smape:.2f}%")

    # Statistical significance: AttentionBiLSTM vs others (RMSE per fold)
    if SCIPY_AVAILABLE:
        print("\n  >>> STATISTICAL SIGNIFICANCE TESTS (RMSE, Paired t-test)")
        att_name = "AttentionBiLSTM"
        for baseline_conf in MODELS[1:]:
            base_name = baseline_conf["name"]
            print(f"\n    Comparing {att_name} vs {base_name}")
            for t_name in TARGET_FEATURES:
                att_rmse = np.array(metrics[att_name][t_name]["rmse"])
                base_rmse = np.array(metrics[base_name][t_name]["rmse"])
                min_len = min(len(att_rmse), len(base_rmse))
                if min_len < 2:
                    print(f"      {t_name}: Not enough folds for t-test.")
                    continue
                att_rmse = att_rmse[:min_len]
                base_rmse = base_rmse[:min_len]
                stat, p_val = ttest_rel(att_rmse, base_rmse)
                print(f"      {t_name}: p-value = {p_val:.4f} "
                      f"(mean {att_name} RMSE = {att_rmse.mean():.4f}, "
                      f"mean {base_name} RMSE = {base_rmse.mean():.4f})")
    else:
        print("\n  (SciPy not available, skipping t-test significance analysis.)")

print("\nAll datasets and models processed. Use these metrics and p-values "
      "in your paper's results, comparison tables, and discussion.")



DATASET: Local Microgrid Dataset
Original shape (after basic cleaning): (3546, 17)
Prepared data shape: (3378, 22) (rows x features)
Total points: 3378
Reserved for validation: 1013
Initial training length: 2365
Per-fold test size (approx): 202

  --- Fold 1/5 ---
    Train samples: 2329, Test samples: 203
    -> Training model: AttentionBiLSTM
      AttentionBiLSTM Fold 1 Results:
        solar_pv_output: RMSE = 29.4170, MAE = 25.5524, nRMSE = 0.6151, MAPE = 2413.91%, sMAPE = 61.39%
        wind_power_output: RMSE = 29.6469, MAE = 26.1019, nRMSE = 0.6288, MAPE = 235.77%, sMAPE = 62.69%
        grid_load_demand: RMSE = 126.7650, MAE = 108.4341, nRMSE = 0.4686, MAPE = 68.03%, sMAPE = 43.86%
    -> Training model: BiLSTM
      BiLSTM Fold 1 Results:
        solar_pv_output: RMSE = 29.2521, MAE = 25.4373, nRMSE = 0.6116, MAPE = 2385.69%, sMAPE = 61.41%
        wind_power_output: RMSE = 29.6801, MAE = 26.0654, nRMSE = 0.6295, MAPE = 231.61%, sMAPE = 62.78%
        grid_load_demand: RMSE =

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


      SimpleLSTM Fold 1 Results:
        solar_pv_output: RMSE = 29.3870, MAE = 25.5692, nRMSE = 0.6145, MAPE = 2443.24%, sMAPE = 61.61%
        wind_power_output: RMSE = 29.6621, MAE = 26.0355, nRMSE = 0.6291, MAPE = 231.50%, sMAPE = 62.68%
        grid_load_demand: RMSE = 127.9645, MAE = 108.9728, nRMSE = 0.4730, MAPE = 68.38%, sMAPE = 44.02%

  --- Fold 2/5 ---
    Train samples: 2531, Test samples: 203
    -> Training model: AttentionBiLSTM
      AttentionBiLSTM Fold 2 Results:
        solar_pv_output: RMSE = 30.3829, MAE = 26.3586, nRMSE = 0.5991, MAPE = 20152.67%, sMAPE = 60.16%
        wind_power_output: RMSE = 28.4131, MAE = 23.7624, nRMSE = 0.5494, MAPE = 166.09%, sMAPE = 53.52%
        grid_load_demand: RMSE = 130.7516, MAE = 115.2998, nRMSE = 0.5142, MAPE = 75.62%, sMAPE = 48.81%
    -> Training model: BiLSTM
      BiLSTM Fold 2 Results:
        solar_pv_output: RMSE = 30.3539, MAE = 26.3302, nRMSE = 0.5985, MAPE = 21227.66%, sMAPE = 59.66%
        wind_power_output: RMSE = 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


      SimpleLSTM Fold 2 Results:
        solar_pv_output: RMSE = 30.2394, MAE = 26.1829, nRMSE = 0.5963, MAPE = 20493.76%, sMAPE = 59.52%
        wind_power_output: RMSE = 28.3452, MAE = 23.7733, nRMSE = 0.5481, MAPE = 171.35%, sMAPE = 53.28%
        grid_load_demand: RMSE = 130.4113, MAE = 114.5219, nRMSE = 0.5129, MAPE = 77.59%, sMAPE = 48.37%

  --- Fold 3/5 ---
    Train samples: 2733, Test samples: 203
    -> Training model: AttentionBiLSTM
      AttentionBiLSTM Fold 3 Results:
        solar_pv_output: RMSE = 28.8438, MAE = 25.3189, nRMSE = 0.5691, MAPE = 251.57%, sMAPE = 57.93%
        wind_power_output: RMSE = 28.8346, MAE = 25.1393, nRMSE = 0.5968, MAPE = 266.26%, sMAPE = 59.51%
        grid_load_demand: RMSE = 127.5754, MAE = 111.9483, nRMSE = 0.4651, MAPE = 66.43%, sMAPE = 44.86%
    -> Training model: BiLSTM
      BiLSTM Fold 3 Results:
        solar_pv_output: RMSE = 28.7851, MAE = 25.1951, nRMSE = 0.5680, MAPE = 255.07%, sMAPE = 57.57%
        wind_power_output: RMSE = 28.

...

      SimpleLSTM Fold 3 Results:
        solar_pv_output: RMSE = 1295.7091, MAE = 595.5432, nRMSE = 0.6077, MAPE = 415.90%, sMAPE = 134.73%
        wind_power_output: RMSE = 4842.6612, MAE = 3282.5868, nRMSE = 0.2808, MAPE = 24.49%, sMAPE = 23.04%
        grid_load_demand: RMSE = 1888.2448, MAE = 1140.4303, nRMSE = 0.0333, MAPE = 2.12%, sMAPE = 2.09%

  --- Fold 4/5 ---
    Train samples: 44161, Test samples: 3014
    -> Training model: AttentionBiLSTM
      AttentionBiLSTM Fold 4 Results:
        solar_pv_output: RMSE = 2252.4275, MAE = 1159.4603, nRMSE = 0.3847, MAPE = 581.62%, sMAPE = 109.87%
        wind_power_output: RMSE = 5277.7272, MAE = 3689.8743, nRMSE = 0.2918, MAPE = 37.48%, sMAPE = 28.07%
        grid_load_demand: RMSE = 2228.2001, MAE = 1613.7916, nRMSE = 0.0411, MAPE = 3.02%, sMAPE = 2.99%
    -> Training model: BiLSTM
      BiLSTM Fold 4 Results:
        solar_pv_output: RMSE = 2307.7691, MAE = 1174.1244, nRMSE = 0.3942, MAPE = 730.11%, sMAPE = 110.02%
        wind_powe

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


      SimpleLSTM Fold 4 Results:
        solar_pv_output: RMSE = 2230.9713, MAE = 1088.6396, nRMSE = 0.3811, MAPE = 523.98%, sMAPE = 108.37%
        wind_power_output: RMSE = 5232.6705, MAE = 3605.3077, nRMSE = 0.2893, MAPE = 34.88%, sMAPE = 27.75%
        grid_load_demand: RMSE = 2162.6445, MAE = 1406.9932, nRMSE = 0.0399, MAPE = 2.67%, sMAPE = 2.65%

  --- Fold 5/5 ---
    Train samples: 47174, Test samples: 3014
    -> Training model: AttentionBiLSTM
      AttentionBiLSTM Fold 5 Results:
        solar_pv_output: RMSE = 2353.7712, MAE = 1339.8448, nRMSE = 0.3084, MAPE = 449.65%, sMAPE = 90.57%
        wind_power_output: RMSE = 3953.1630, MAE = 2663.7885, nRMSE = 0.4360, MAPE = 46.13%, sMAPE = 34.27%
        grid_load_demand: RMSE = 1306.0651, MAE = 967.8251, nRMSE = 0.0257, MAPE = 1.92%, sMAPE = 1.91%
    -> Training model: BiLSTM
      BiLSTM Fold 5 Results:
        solar_pv_output: RMSE = 2267.8307, MAE = 1228.1017, nRMSE = 0.2972, MAPE = 459.06%, sMAPE = 89.01%
        wind_power_

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


      SimpleLSTM Fold 5 Results:
        solar_pv_output: RMSE = 2052.3209, MAE = 1098.3205, nRMSE = 0.2689, MAPE = 424.97%, sMAPE = 87.03%
        wind_power_output: RMSE = 3739.8234, MAE = 2460.9184, nRMSE = 0.4125, MAPE = 39.16%, sMAPE = 34.58%
        grid_load_demand: RMSE = 1469.1412, MAE = 982.6902, nRMSE = 0.0289, MAPE = 1.95%, sMAPE = 1.95%

  >>> FINAL METRICS FOR DATASET: OPSD Germany Dataset (DE)

    MODEL: AttentionBiLSTM
      SOLAR_PV_OUTPUT
        Average RMSE       : 2057.3802
        Stability (Std Dev): 383.6136
        Average nRMSE      : 0.4155
        Average MAPE       : 437.73%
        Average sMAPE      : 108.32%
      WIND_POWER_OUTPUT
        Average RMSE       : 4520.3987
        Stability (Std Dev): 649.1186
        Average nRMSE      : 0.3399
        Average MAPE       : 35.59%
        Average sMAPE      : 29.03%
      GRID_LOAD_DEMAND
        Average RMSE       : 1773.0157
        Stability (Std Dev): 313.0438
        Average nRMSE      : 0.0324
      